In [ ]:
import pandas as pd
import requests
import os

data_path = "../../data"

# COSMIC

In [ ]:
gene_table = pd.read_csv(f'{data_path}/Census_allFri Nov 17 10_27_09 2023.tsv', sep='\t')
gene_table.head()

In [ ]:
def get_gene_position(gene_name):
    server = "https://grch37.rest.ensembl.org"
    ext = f"/lookup/symbol/homo_sapiens/{gene_name}?content-type=application/json"
    r = requests.get(server+ext, headers={"Content-Type": "application/json"})
    
    if not r.ok:
        return None, None, None

    decoded = r.json()
    return decoded['seq_region_name'], decoded['start'], decoded['end']

# Apply the function to each gene in gene_table
gene_table['chrom'], gene_table['start'], gene_table['end'] = zip(*gene_table['Gene Symbol'].map(get_gene_position))

In [ ]:
genes = gene_table[['Gene Symbol', 'chrom', 'start', 'end']]
genes.head()

In [ ]:
# find rows with nan
nan_rows = genes[genes["start"].isna()]
len(nan_rows)

In [ ]:
genes = genes[~genes["start"].isna()]
genes["start"] = genes["start"].astype(int)
genes["end"] = genes["end"].astype(int)
genes.sort_values(by=["chrom", "start"], inplace=True)
genes["chrom"] = "chr" + genes["chrom"]
genes.rename(columns={"Gene Symbol": "gene"}, inplace=True)

In [ ]:
genes.to_csv(f'{data_path}/COSMIC_consensus_genes.tsv', sep="\t", index=False)

## PCAGW

The sample sheet is downloaded from https://dcc.icgc.org/releases/PCAWG/donors_and_biospecimens


The CN files are obtained by donwloading both `consensus.20170119.somatic.cns.icgc.public.tar.gz` and ` consensus.20170119.somatic.cna.tcga.public.tar.gz` from https://dcc.icgc.org/releases/PCAWG/consensus_cnv . 

In [ ]:
pcawg_ids = pd.read_csv('PCAWG/pcawg_sample_sheet.tsv', sep='\t')

In [ ]:
directory = 'PCAWG'

pcawg_samples = []

for filename in os.listdir(directory):
    if filename.endswith(".txt"):
        df = pd.read_csv(os.path.join(directory, filename), sep='\t')   
        sample_id = filename[0:filename.find('.')]     
        df.insert(0, 'aliquot_id', sample_id)        
        pcawg_samples.append(df)
len(pcawg_samples)

In [ ]:
pcawg_dfs = pd.concat(pcawg_samples)
pcawg_dfs.head()

In [ ]:
mapping = pcawg_ids[["aliquot_id", "icgc_specimen_id"]]
map_dict = mapping.set_index('aliquot_id')['icgc_specimen_id'].to_dict()
pcawg_dfs['aliquot_id'] = pcawg_dfs['aliquot_id'].map(map_dict)
pcawg_dfs.head()

In [ ]:
pcawg_dfs = pcawg_dfs.rename(columns={'aliquot_id': 'sample_id', 'chromosome': 'chrom'})
pcawg_dfs['start'] = pcawg_dfs['start'].astype(int)
pcawg_dfs['end'] = pcawg_dfs['end'].astype(int)
pcawg_dfs['chrom'] = 'chr' + pcawg_dfs['chrom'].astype(str)
# drop star
pcawg_dfs = pcawg_dfs.drop(columns=['star', "total_cn"])

In [ ]:
# count rows in final_df that have NaN values
pcawg_dfs[pcawg_dfs.isna().any(axis=1)].shape[0]

In [ ]:
pcawg_dfs.to_csv(f'{data_path}/pcawg_cns_source.tsv', sep='\t', index=False)

## TGCA hg19

The CN files are in the archive: https://github.com/VanLoo-lab/ascat/raw/master/ReleasedData/TCGA_SNP6_hg19/segments.tar.gz .

In [ ]:
directory = 'tcga_hg19'

tcga_hg19_samples = []

for filename in os.listdir(directory):
    if filename.endswith(".txt"):
        df = pd.read_csv(os.path.join(directory, filename), sep='\t')      
        tcga_hg19_samples.append(df)
len(tcga_hg19_samples)

In [ ]:
tcga_hg19_dfs = pd.concat(tcga_hg19_samples)
tcga_hg19_dfs.head()

In [ ]:
tcga_hg19_dfs.columns = ["sample_id", "chrom", "start", "end", "major_cn", "minor_cn"]
tcga_hg19_dfs['chrom'] = 'chr' + tcga_hg19_dfs['chrom'].astype(str)
tcga_hg19_dfs.head()

In [ ]:
tcga_hg19_dfs.to_csv(f'{data_path}/tcga_hg19_cns_source.tsv', sep='\t', index=False)

## TGCA hg38

The CN files are in the archive: https://github.com/VanLoo-lab/ascat/raw/master/ReleasedData/TCGA_SNP6_hg38/segments.tar.gz .

In [ ]:
directory = 'tcga_hg38'

tcga_hg38_samples = []

for filename in os.listdir(directory):
    if filename.endswith(".txt"):
        df = pd.read_csv(os.path.join(directory, filename), sep='\t')      
        tcga_hg38_samples.append(df)
len(tcga_hg38_samples)

In [ ]:
tcga_hg38_dfs = pd.concat(tcga_hg38_samples)
tcga_hg38_dfs.head()

In [ ]:
tcga_hg38_dfs.columns = ["sample_id", "chrom", "start", "end", "major_cn", "minor_cn"]
tcga_hg38_dfs['chrom'] = 'chr' + tcga_hg38_dfs['chrom'].astype(str)
tcga_hg38_dfs.head()

In [ ]:
tcga_hg38_dfs.to_csv(f'{data_path}/tcga_hg38_cns_source.tsv', sep='\t', index=False)

# ENSAMBLE

In [ ]:
# Install the assembly if missing
# pip install pyensembl
# !pyensembl install --release 75 --species homo_sapiens

from pyensembl import EnsemblRelease

In [ ]:
# Specify the Ensembl release
data = EnsemblRelease(75)

genes = data.genes()

contig_names = list(map(str, range(1, 23))) + ['X', 'Y']

# Create a list of gene positions
gene_positions = [(gene.gene_id, "chr" + gene.contig, gene.start, gene.end) for gene in genes if gene.biotype == 'protein_coding' and gene.contig in contig_names]

pos_df = pd.DataFrame(gene_positions, columns=["gene", "chrom", "start", "end"])
pos_df.head()

In [ ]:
pos_df.to_csv(f'{data_path}/ENSEMBL_coding.tsv', index=False, sep="\t")